# exp37 — planted-backdoor detection on Pythia-160M (transformer-scale evidence)

Traceprop-only (no LogIX). Plants a trigger into K training examples (label flipped to a
target class) + distractors (trigger, correct label), trains with inline capture, and checks
whether attribution ranks the poisoned examples via **auc_within_target** (AUC over target-label
candidates only — isolates the trigger from the label-sign artifact). Clones `main`.

## Cell 1 — GPU check + mount Drive

In [ ]:
import subprocess
gpu = subprocess.run(["nvidia-smi","--query-gpu=name","--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "NONE")
assert any(g in gpu for g in ("A100","L4")), f"Need A100/L4, got {gpu!r}"
from google.colab import drive
drive.mount('/content/drive')

## Cell 2 — Clone main + install deps

In [ ]:
%cd /content
!rm -rf Traceprop && git clone -q https://github.com/AmitoVrito/Traceprop.git
%cd /content/Traceprop
!git checkout -q main
!git log --oneline -1
import subprocess as _sp
_fix = _sp.run(["grep","-q","_tok.pad_token_id","experiments/exp27_lds_quality.py"]).returncode == 0
print("PAD-TOKEN FIX PRESENT:", _fix)
assert _fix, "STALE CLONE -- re-open the notebook fresh from GitHub"
!pip -q install "datasets>=2.14" "peft>=0.7" scikit-learn
!pip -q uninstall -y torchao   # Colab's torchao 0.10 breaks peft 0.20 LoRA dispatch
!pip -q install -e . --no-deps
import torch, transformers, peft, sklearn, datasets
print(f"torch {torch.__version__} | transformers {transformers.__version__} | peft {peft.__version__} | sklearn {sklearn.__version__}")
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## Cell 3 — Run exp37 (Pythia-160M, SST-2, 3 seeds) + save to Drive

In [ ]:
import subprocess, sys, os, json, shutil
os.chdir('/content/Traceprop/experiments')
cmd = [sys.executable, "exp37_planted_detection.py",
       "--backend", "hf", "--model", "EleutherAI/pythia-160m", "--data", "sst2",
       "--device", "cuda", "--gpu_check", "", "--track", "0",
       "--n_train", "2000", "--seq", "64", "--epochs", "10", "--lr", "2e-4", "--batch", "16",
       "--plant_frac", "0.1", "--distractor_frac", "0.05", "--trigger_len", "8",
       "--backdoor_target_label", "1", "--n_seeds", "3",
       "--out", "results/exp37_pythia160m_sst2.json", "--force"]
p = subprocess.run(cmd, capture_output=True, text=True)
os.makedirs("/content/drive/MyDrive/traceprop_results/", exist_ok=True)
open("/content/drive/MyDrive/traceprop_results/exp37_stderr.txt","w").write(p.stderr)
open("/content/drive/MyDrive/traceprop_results/exp37_stdout.txt","w").write(p.stdout)
if p.returncode != 0:
    print("EXIT", p.returncode, "STDERR tail:\n" + "\n".join(p.stderr.splitlines()[-45:]))
    raise SystemExit("exp37 failed")
# print the key lines: backdoor learned? + auc_within_target per method
for ln in p.stdout.splitlines():
    if any(k in ln for k in ("backdoor_success", "backdoor-", "learned", "NO seed", "WARNING")):
        print(ln)
for f in ("results/exp37_pythia160m_sst2.json", "results/exp37_pythia160m_sst2_raw.npz"):
    if os.path.exists(f): shutil.copy(f, "/content/drive/MyDrive/traceprop_results/")
print("saved to Drive")